Adaptive Graph Retrieval

Goal: Can an adaptive retrieval strategy retrieve more relevant and complete graph evidence than fixed 0-hop, 1-hop, 2-hop, and manually selected relationship-aware retrieval?

In [1]:
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase
import pandas as pd
import os
import time
import statistics

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env")

DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(
        os.getenv("NEO4J_USER"),
        os.getenv("NEO4J_PASSWORD")
    )
)

driver.verify_connectivity()

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


In [2]:
records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    RETURN count(n) AS nodes
    """,
    database_=DATABASE
)

node_count = records[0]["nodes"]

records, _, _ = driver.execute_query(
    """
    MATCH (:Entity)-[r]->(:Entity)
    RETURN count(r) AS relationships
    """,
    database_=DATABASE
)

relationship_count = records[0]["relationships"]

print("Entity nodes:", node_count)
print("Relationships:", relationship_count)

Entity nodes: 25
Relationships: 27


Goal is to retrieve only the relevant evidence for a Sinhala question, rather than retrieving every connected node.

Define adaptive retrieval queries

In [3]:
ADAPTIVE_QUERIES = {
    "height": """
        MATCH (a:Attraction {id: $entity_id})
        WHERE a.height_m IS NOT NULL
        RETURN a.id AS entity_id,
               a.name_si AS name,
               a.height_m AS height_m,
               a.height_approximate AS approximate
    """,

    "visiting_time": """
        MATCH (a:Attraction {id: $entity_id})
        WHERE a.best_time_si IS NOT NULL
        RETURN a.id AS entity_id,
               a.name_si AS name,
               a.best_time_si AS visiting_time
    """,

    "district": """
        MATCH (d:Destination {id: $entity_id})
              -[:LOCATED_IN]->(district:District)
        RETURN district.id AS entity_id,
               district.name_si AS district
    """,

    "province": """
        MATCH (d:Destination {id: $entity_id})
              -[:LOCATED_IN]->(district:District)
              -[:PART_OF]->(province:Province)
        RETURN district.name_si AS district,
               province.name_si AS province
    """,

    "accommodation": """
        MATCH (d:Destination {id: $entity_id})
              -[:HAS_ACCOMMODATION]->(a:Accommodation)
        RETURN a.id AS entity_id,
               a.name_si AS accommodation
        ORDER BY accommodation
    """,

    "hiking": """
        MATCH (d:Destination {id: $entity_id})
              -[:HAS_ATTRACTION]->(a:Attraction)
              -[:SUPPORTS_ACTIVITY]->(:Activity {id: "AC001"})
        RETURN a.id AS entity_id,
               a.name_si AS hiking_location
        ORDER BY hiking_location
    """,

    "train_services": """
        MATCH (d:Destination {id: $entity_id})
              -[:HAS_RAILWAY_STATION]->(s:RailwayStation)
              <-[:ARRIVES_AT]-(t:TrainService)
        RETURN t.id AS entity_id,
               t.name_si AS train,
               t.train_number AS train_number
        ORDER BY train_number
    """,

    "height_comparison": """
        MATCH (a:Attraction {id: $entity_id})
        MATCH (b:Attraction {id: $entity_id_2})
        WHERE a.height_m IS NOT NULL
          AND b.height_m IS NOT NULL
        RETURN a.id AS first_id,
               a.name_si AS first_name,
               a.height_m AS first_height,
               b.id AS second_id,
               b.name_si AS second_name,
               b.height_m AS second_height
    """
}

print("Adaptive query templates:", len(ADAPTIVE_QUERIES))

Adaptive query templates: 8


Create the adaptive retrieval function

In [4]:
def adaptive_retrieve(intent, entity_ids):

    if intent not in ADAPTIVE_QUERIES:
        return {
            "status": "unsupported",
            "records": [],
            "method": "adaptive"
        }

    if intent == "height_comparison":
        if len(entity_ids) != 2:
            return {
                "status": "missing_entities",
                "records": [],
                "method": "adaptive"
            }

        params = {
            "entity_id": entity_ids[0],
            "entity_id_2": entity_ids[1]
        }

    else:
        if len(entity_ids) != 1:
            return {
                "status": "missing_or_ambiguous_entities",
                "records": [],
                "method": "adaptive"
            }

        params = {"entity_id": entity_ids[0]}

    query = ADAPTIVE_QUERIES[intent]

    records, _, _ = driver.execute_query(
        query,
        parameters_=params,
        database_=DATABASE
    )

    return {
        "status": "success" if records else "no_evidence",
        "method": "adaptive",
        "intent": intent,
        "entity_ids": entity_ids,
        "records": [dict(r) for r in records]
    }

Test the adaptive function

In [5]:
test_cases = [
    ("province", ["D001"]),
    ("accommodation", ["D001"]),
    ("hiking", ["D001"]),
    ("height", ["A004"]),
    ("height_comparison", ["A004", "A007"]),
    ("train_services", ["D001"]),
    ("visiting_time", ["A001"])
]

for intent, entity_ids in test_cases:
    print("\n" + "=" * 60)
    print("Intent:", intent)
    print("Entities:", entity_ids)

    result = adaptive_retrieve(intent, entity_ids)

    print("Status:", result["status"])

    for record in result["records"]:
        print(record)


Intent: province
Entities: ['D001']
Status: success
{'district': 'බදුල්ල', 'province': 'ඌව පළාත'}

Intent: accommodation
Entities: ['D001']
Status: success
{'entity_id': 'H001', 'accommodation': '98 Acres Resort & Spa'}
{'entity_id': 'H003', 'accommodation': 'Ella Jungle Resort'}
{'entity_id': 'H004', 'accommodation': 'La Ella Breeze'}
{'entity_id': 'H002', 'accommodation': 'The Secret Ella'}

Intent: hiking
Entities: ['D001']
Status: success
{'entity_id': 'A002', 'hiking_location': 'ඇල්ල රොක්'}
{'entity_id': 'A001', 'hiking_location': 'පුංචි ශ්\u200dරී පාදය'}

Intent: height
Entities: ['A004']
Status: success
{'entity_id': 'A004', 'name': 'රාවණා ඇල්ල', 'height_m': 25, 'approximate': True}

Intent: height_comparison
Entities: ['A004', 'A007']
Status: success
{'first_id': 'A004', 'first_name': 'රාවණා ඇල්ල', 'first_height': 25, 'second_id': 'A007', 'second_name': 'දියලුම දිය ඇල්ල', 'second_height': 220}

Intent: train_services
Entities: ['D001']
Status: success
{'entity_id': 'T001', 'tr

measuring whether adaptive retrieval is actually better than the other approaches.

Create an evidence retrieval function

In [6]:
def retrieve_graph_evidence(entity_ids, max_hops=2):
    """
    Retrieve graph relationships and node properties
    within a specified traversal depth.
    """

    query = """
    MATCH (start:Entity)
    WHERE start.id IN $entity_ids

    OPTIONAL MATCH path = (start)-[*0..2]-(connected:Entity)
    WHERE length(path) <= $max_hops

    WITH collect(DISTINCT path) AS paths

    UNWIND paths AS path

    WITH
        nodes(path) AS path_nodes,
        relationships(path) AS path_relationships

    UNWIND path_nodes AS n

    WITH
        collect(DISTINCT n) AS all_nodes,
        path_relationships

    UNWIND all_nodes AS node

    RETURN DISTINCT
        node.id AS entity_id,
        node.name_si AS entity_name,
        properties(node) AS properties
    """

    records, _, _ = driver.execute_query(
        query,
        parameters_={
            "entity_ids": entity_ids,
            "max_hops": max_hops
        },
        database_=DATABASE
    )

    return [dict(record) for record in records]

In [7]:
evidence = retrieve_graph_evidence(["D001"], max_hops=2)

print("Retrieved nodes:", len(evidence))

for record in evidence[:5]:
    print(record)

Retrieved nodes: 25
{'entity_id': 'D001', 'entity_name': 'ඇල්ල', 'properties': {'elevation_m': 1041, 'climate_si': 'සාමාන්\u200dයයෙන් සිසිල් කඳුකර දේශගුණය', 'name_si': 'ඇල්ල', 'source_document': 'ella.txt', 'id': 'D001', 'source_section': '1, 2'}}
{'entity_id': 'DI001', 'entity_name': 'බදුල්ල', 'properties': {'name_si': 'බදුල්ල', 'id': 'DI001'}}
{'entity_id': 'P001', 'entity_name': 'ඌව පළාත', 'properties': {'name_si': 'ඌව පළාත', 'id': 'P001'}}
{'entity_id': 'A001', 'entity_name': 'පුංචි ශ්\u200dරී පාදය', 'properties': {'elevation_m': 1141, 'aliases': ['පුංචි ශ්\u200dරී පාදය', "Little Adam's Peak", "Mini Adam's Peak"], 'name_si': 'පුංචි ශ්\u200dරී පාදය', 'visiting_advice_si': 'උදෑසන සහ සවස් කාලය ජනප්\u200dරියය.', 'hiking_difficulty': 'easy', 'description_si': 'ඇල්ලේ ජනප්\u200dරිය කඳු නැගීමේ ස්ථානයකි.', 'elevation_approximate': True, 'best_time_si': 'උදෑසන සහ සවස් කාලය', 'source_document': 'ella.txt', 'id': 'A001', 'source_sections': [3], 'source_section': '3', 'name_en': "Little Adam's 

Define the normalized graph evidence function

In [8]:
def get_graph_evidence(entity_ids, max_hops=2):
    """
    Read-only graph evidence extraction.

    Returns:
      node facts: (entity_id, property_name, property_value)
      relationship facts: (source_id, relationship_type, target_id)

    Uses undirected traversal for discovery, but preserves
    the original direction of each relationship.
    """

    if max_hops not in (0, 1, 2):
        raise ValueError("max_hops must be 0, 1, or 2")

    query = f"""
    MATCH (start:Entity)
    WHERE start.id IN $entity_ids

    MATCH path = (start)-[*0..{max_hops}]-(n:Entity)

    WITH collect(DISTINCT n) AS reached_nodes,
         collect(DISTINCT relationships(path)) AS relationship_lists

    UNWIND reached_nodes AS node

    RETURN DISTINCT
        node.id AS entity_id,
        properties(node) AS node_properties
    """

    # Simpler, explicit query to avoid relationship aggregation issues
    node_query = f"""
    MATCH (start:Entity)
    WHERE start.id IN $entity_ids

    MATCH p = (start)-[*0..{max_hops}]-(n:Entity)

    RETURN DISTINCT
        n.id AS entity_id,
        properties(n) AS node_properties,
        length(p) AS distance
    """

    relationship_query = f"""
    MATCH (start:Entity)
    WHERE start.id IN $entity_ids

    MATCH p = (start)-[*0..{max_hops}]-(n:Entity)

    UNWIND relationships(p) AS rel

    RETURN DISTINCT
        startNode(rel).id AS source_id,
        type(rel) AS relationship_type,
        endNode(rel).id AS target_id
    """

    node_records, _, _ = driver.execute_query(
        node_query,
        parameters_={"entity_ids": entity_ids},
        database_=DATABASE
    )

    relationship_records, _, _ = driver.execute_query(
        relationship_query,
        parameters_={"entity_ids": entity_ids},
        database_=DATABASE
    )

    min_distances = {}

    for record in node_records:
        entity_id = record["entity_id"]
        distance = record["distance"]

        if entity_id not in min_distances:
            min_distances[entity_id] = distance
        else:
            min_distances[entity_id] = min(
                min_distances[entity_id], distance
            )

    node_properties = {}
    for record in node_records:
        node_properties[record["entity_id"]] = (
            record["node_properties"]
        )

    node_facts = []

    metadata_keys = {
        "id", "name_si", "name_en", "aliases",
        "source_document", "source_section", "source_sections"
    }

    for entity_id, properties in node_properties.items():
        for key, value in properties.items():
            if key not in metadata_keys:
                node_facts.append({
                    "kind": "property",
                    "entity_id": entity_id,
                    "property": key,
                    "value": value
                })

    relationship_facts = [
        {
            "kind": "relationship",
            "source_id": record["source_id"],
            "relationship": record["relationship_type"],
            "target_id": record["target_id"]
        }
        for record in relationship_records
    ]

    return {
        "node_facts": node_facts,
        "relationship_facts": relationship_facts,
        "distances": min_distances
    }

Test 0-hop, 1-hop and 2-hop evidence

In [9]:
for hops in [0, 1, 2]:

    evidence = get_graph_evidence(["D001"], max_hops=hops)

    print("\n" + "=" * 50)
    print("Retrieval depth:", hops)
    print("Node property facts:", len(evidence["node_facts"]))
    print("Relationship facts:", len(evidence["relationship_facts"]))
    print("Entities reached:", len(evidence["distances"]))

    print("\nSample relationships:")
    for fact in evidence["relationship_facts"][:5]:
        print(fact)


Retrieval depth: 0
Node property facts: 2
Relationship facts: 0
Entities reached: 1

Sample relationships:

Retrieval depth: 1
Node property facts: 35
Relationship facts: 19
Entities reached: 20

Sample relationships:
{'kind': 'relationship', 'source_id': 'D001', 'relationship': 'LOCATED_IN', 'target_id': 'DI001'}
{'kind': 'relationship', 'source_id': 'D001', 'relationship': 'HAS_ATTRACTION', 'target_id': 'A001'}
{'kind': 'relationship', 'source_id': 'D001', 'relationship': 'HAS_ATTRACTION', 'target_id': 'A002'}
{'kind': 'relationship', 'source_id': 'D001', 'relationship': 'HAS_ATTRACTION', 'target_id': 'A003'}
{'kind': 'relationship', 'source_id': 'D001', 'relationship': 'HAS_ATTRACTION', 'target_id': 'A004'}

Retrieval depth: 2
Node property facts: 48
Relationship facts: 27
Entities reached: 25

Sample relationships:
{'kind': 'relationship', 'source_id': 'D001', 'relationship': 'LOCATED_IN', 'target_id': 'DI001'}
{'kind': 'relationship', 'source_id': 'DI001', 'relationship': 'PART_O

Standardize fact identifiers

In [11]:
def relationship_fact(source, relation, target):
    return ("relationship", source, relation, target)


def property_fact(entity, property_name, value):
    return ("property", entity, property_name, str(value))


def normalize_evidence(evidence):
    facts = set()

    for item in evidence["relationship_facts"]:
        facts.add(
            relationship_fact(
                item["source_id"],
                item["relationship"],
                item["target_id"]
            )
        )

    for item in evidence["node_facts"]:
        facts.add(
            property_fact(
                item["entity_id"],
                item["property"],
                item["value"]
            )
        )

    return facts

Define required evidence for six questions

In [12]:
GOLD_EVIDENCE = {
    "Q001": {
        "question": "ඇල්ල පිහිටා තිබෙන දිස්ත්‍රික්කය අයත් වන පළාත කුමක්ද?",
        "intent": "province",
        "entities": ["D001"],
        "facts": {
            relationship_fact("D001", "LOCATED_IN", "DI001"),
            relationship_fact("DI001", "PART_OF", "P001")
        }
    },

    "Q002": {
        "question": "ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?",
        "intent": "accommodation",
        "entities": ["D001"],
        "facts": {
            relationship_fact("D001", "HAS_ACCOMMODATION", "H001"),
            relationship_fact("D001", "HAS_ACCOMMODATION", "H002"),
            relationship_fact("D001", "HAS_ACCOMMODATION", "H003"),
            relationship_fact("D001", "HAS_ACCOMMODATION", "H004")
        }
    },

    "Q003": {
        "question": "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?",
        "intent": "hiking",
        "entities": ["D001"],
        "facts": {
            relationship_fact("D001", "HAS_ATTRACTION", "A001"),
            relationship_fact("D001", "HAS_ATTRACTION", "A002"),
            relationship_fact("A001", "SUPPORTS_ACTIVITY", "AC001"),
            relationship_fact("A002", "SUPPORTS_ACTIVITY", "AC001")
        }
    },

    "Q004": {
        "question": "රාවණා ඇල්ලේ උස කීයද?",
        "intent": "height",
        "entities": ["A004"],
        "facts": {
            property_fact("A004", "height_m", 25)
        }
    },

    "Q005": {
        "question": "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
        "intent": "height_comparison",
        "entities": ["A004", "A007"],
        "facts": {
            property_fact("A004", "height_m", 25),
            property_fact("A007", "height_m", 220)
        }
    },

    "Q006": {
        "question": "පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?",
        "intent": "visiting_time",
        "entities": ["A001"],
        "facts": {
            property_fact(
                "A001",
                "best_time_si",
                "උදෑසන සහ සවස් කාලය"
            )
        }
    }
}

print("Gold-standard questions:", len(GOLD_EVIDENCE))

Gold-standard questions: 6


Evaluate 0-hop, 1-hop, and 2-hop

In [13]:
def score_evidence(retrieved_facts, required_facts):
    matched = retrieved_facts & required_facts

    precision = (
        len(matched) / len(retrieved_facts)
        if retrieved_facts else 0.0
    )

    recall = (
        len(matched) / len(required_facts)
        if required_facts else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "complete": required_facts.issubset(retrieved_facts),
        "retrieved_count": len(retrieved_facts),
        "matched_count": len(matched)
    }


evaluation_rows = []

for question_id, item in GOLD_EVIDENCE.items():
    for depth in [0, 1, 2]:

        evidence = get_graph_evidence(
            item["entities"],
            max_hops=depth
        )

        facts = normalize_evidence(evidence)

        scores = score_evidence(
            retrieved_facts=facts,
            required_facts=item["facts"]
        )

        evaluation_rows.append({
            "Question_ID": question_id,
            "Method": f"{depth}-hop",
            **scores
        })

evidence_eval_df = pd.DataFrame(evaluation_rows)

display(evidence_eval_df)

,Question_ID,Method,precision,recall,f1,complete,retrieved_count,matched_count
0,Q001,0-hop,0.000000,0.0,0.000000,False,2,0
1,Q001,1-hop,0.018519,0.5,0.035714,False,54,1
2,Q001,2-hop,0.026667,1.0,0.051948,True,75,2
3,Q002,0-hop,0.000000,0.0,0.000000,False,2,0
4,Q002,1-hop,0.074074,1.0,0.137931,True,54,4
5,Q002,2-hop,0.053333,1.0,0.101266,True,75,4
6,Q003,0-hop,0.000000,0.0,0.000000,False,2,0
7,Q003,1-hop,0.037037,0.5,0.068966,False,54,2
8,Q003,2-hop,0.053333,1.0,0.101266,True,75,4
9,Q004,0-hop,0.200000,1.0,0.333333,True,5,1


Compare average performance

In [14]:
summary_df = (
    evidence_eval_df
    .groupby("Method", as_index=False)
    .agg(
        Mean_Precision=("precision", "mean"),
        Mean_Recall=("recall", "mean"),
        Mean_F1=("f1", "mean"),
        Complete_Questions=("complete", "sum"),
        Mean_Facts=("retrieved_count", "mean")
    )
)

display(summary_df.round(3))

,Method,Mean_Precision,Mean_Recall,Mean_F1,Complete_Questions,Mean_Facts
0,0-hop,0.103,0.500,0.170,3,4.167
1,1-hop,0.087,0.833,0.155,4,32.000
2,2-hop,0.034,1.000,0.066,6,64.833


Define targeted evidence retrieval

In [15]:
def targeted_evidence_retrieve(intent, entity_ids):
    """
    Retrieve relationship/property evidence relevant to an intent.

    Returns normalized facts:
    ("relationship", source_id, relation, target_id)
    ("property", entity_id, property_name, value)
    """

    facts = set()

    def run(query, params):
        records, _, _ = driver.execute_query(
            query,
            parameters_=params,
            database_=DATABASE
        )
        return records

    if intent == "province":
        query = """
        MATCH (d:Destination {id: $id})
              -[:LOCATED_IN]->(district:District)
              -[:PART_OF]->(province:Province)
        RETURN d.id AS destination,
               district.id AS district,
               province.id AS province
        """

        for r in run(query, {"id": entity_ids[0]}):
            facts.add(relationship_fact(
                r["destination"], "LOCATED_IN", r["district"]
            ))
            facts.add(relationship_fact(
                r["district"], "PART_OF", r["province"]
            ))

    elif intent == "district":
        query = """
        MATCH (d:Destination {id: $id})
              -[:LOCATED_IN]->(district:District)
        RETURN d.id AS destination,
               district.id AS district
        """

        for r in run(query, {"id": entity_ids[0]}):
            facts.add(relationship_fact(
                r["destination"], "LOCATED_IN", r["district"]
            ))

    elif intent == "accommodation":
        query = """
        MATCH (d:Destination {id: $id})
              -[:HAS_ACCOMMODATION]->(a:Accommodation)
        RETURN d.id AS destination,
               a.id AS accommodation
        """

        for r in run(query, {"id": entity_ids[0]}):
            facts.add(relationship_fact(
                r["destination"], "HAS_ACCOMMODATION", r["accommodation"]
            ))

    elif intent == "hiking":
        query = """
        MATCH (d:Destination {id: $id})
              -[:HAS_ATTRACTION]->(a:Attraction)
              -[:SUPPORTS_ACTIVITY]->(:Activity {id: "AC001"})
        RETURN d.id AS destination,
               a.id AS attraction
        """

        for r in run(query, {"id": entity_ids[0]}):
            facts.add(relationship_fact(
                r["destination"], "HAS_ATTRACTION", r["attraction"]
            ))
            facts.add(relationship_fact(
                r["attraction"], "SUPPORTS_ACTIVITY", "AC001"
            ))

    elif intent == "train_services":
        query = """
        MATCH (d:Destination {id: $id})
              -[:HAS_RAILWAY_STATION]->(s:RailwayStation)
              <-[:ARRIVES_AT]-(t:TrainService)
        RETURN d.id AS destination,
               s.id AS station,
               t.id AS train
        """

        for r in run(query, {"id": entity_ids[0]}):
            facts.add(relationship_fact(
                r["destination"], "HAS_RAILWAY_STATION", r["station"]
            ))
            facts.add(relationship_fact(
                r["train"], "ARRIVES_AT", r["station"]
            ))

    elif intent == "height":
        query = """
        MATCH (a:Attraction {id: $id})
        WHERE a.height_m IS NOT NULL
        RETURN a.id AS id, a.height_m AS height
        """

        for r in run(query, {"id": entity_ids[0]}):
            facts.add(property_fact(
                r["id"], "height_m", r["height"]
            ))

    elif intent == "height_comparison":
        query = """
        MATCH (a:Attraction)
        WHERE a.id IN $ids AND a.height_m IS NOT NULL
        RETURN a.id AS id, a.height_m AS height
        """

        for r in run(query, {"ids": entity_ids}):
            facts.add(property_fact(
                r["id"], "height_m", r["height"]
            ))

    elif intent == "visiting_time":
        query = """
        MATCH (a:Attraction {id: $id})
        WHERE a.best_time_si IS NOT NULL
        RETURN a.id AS id, a.best_time_si AS visiting_time
        """

        for r in run(query, {"id": entity_ids[0]}):
            facts.add(property_fact(
                r["id"], "best_time_si", r["visiting_time"]
            ))

    return facts

Test the relationship-aware evidence

In [16]:
for question_id, item in GOLD_EVIDENCE.items():

    facts = targeted_evidence_retrieve(
        item["intent"],
        item["entities"]
    )

    print("\n" + "=" * 55)
    print("Question:", question_id)
    print("Intent:", item["intent"])
    print("Retrieved facts:", len(facts))

    for fact in sorted(facts):
        print(fact)

    print(
        "Gold evidence complete:",
        item["facts"].issubset(facts)
    )


Question: Q001
Intent: province
Retrieved facts: 2
('relationship', 'D001', 'LOCATED_IN', 'DI001')
('relationship', 'DI001', 'PART_OF', 'P001')
Gold evidence complete: True

Question: Q002
Intent: accommodation
Retrieved facts: 4
('relationship', 'D001', 'HAS_ACCOMMODATION', 'H001')
('relationship', 'D001', 'HAS_ACCOMMODATION', 'H002')
('relationship', 'D001', 'HAS_ACCOMMODATION', 'H003')
('relationship', 'D001', 'HAS_ACCOMMODATION', 'H004')
Gold evidence complete: True

Question: Q003
Intent: hiking
Retrieved facts: 4
('relationship', 'A001', 'SUPPORTS_ACTIVITY', 'AC001')
('relationship', 'A002', 'SUPPORTS_ACTIVITY', 'AC001')
('relationship', 'D001', 'HAS_ATTRACTION', 'A001')
('relationship', 'D001', 'HAS_ATTRACTION', 'A002')
Gold evidence complete: True

Question: Q004
Intent: height
Retrieved facts: 1
('property', 'A004', 'height_m', '25')
Gold evidence complete: True

Question: Q005
Intent: height_comparison
Retrieved facts: 2
('property', 'A004', 'height_m', '25')
('property', 'A

Create the adaptive evidence retrieval function

In [17]:
def adaptive_evidence_retrieve(question):
    """
    End-to-end rule-based adaptive retrieval:
    Sinhala question -> intent/entities -> targeted graph evidence.
    """

    analysis = analyze_question(question)

    intent = analysis["intent"]

    entity_ids = [
        entity["id"]
        for entity in analysis["entities"]
    ]

    if intent == "unknown":
        return {
            "status": "unsupported",
            "intent": intent,
            "entity_ids": entity_ids,
            "facts": set()
        }

    required_count = (
        2 if intent == "height_comparison" else 1
    )

    if len(entity_ids) != required_count:
        return {
            "status": "invalid_entities",
            "intent": intent,
            "entity_ids": entity_ids,
            "facts": set()
        }

    facts = targeted_evidence_retrieve(
        intent,
        entity_ids
    )

    return {
        "status": "success" if facts else "no_evidence",
        "intent": intent,
        "entity_ids": entity_ids,
        "facts": facts
    }

Load the functions from  previous notebook

In [22]:
# Sinhala entity aliases
ENTITY_ALIASES = {
    "D001": ["ඇල්ල", "Ella"],
    "A001": ["පුංචි ශ්‍රී පාදය", "පුංචි ශ්\u200dරී පාදය", "Little Adam's Peak"],
    "A002": ["ඇල්ල රොක්", "Ella Rock"],
    "A003": ["ආරුක්කු නවයේ පාලම", "Nine Arches Bridge"],
    "A004": ["රාවණා ඇල්ල", "Ravana Falls"],
    "A005": ["රාවණා ගුහාව", "Ravana Cave"],
    "A006": ["දෝව රජමහා විහාරය"],
    "A007": ["දියලුම ඇල්ල", "දියලුම දිය ඇල්ල", "Diyaluma Falls"],
    "A008": ["නිල්දිය පොකුණ"]
}


def identify_intent(question):
    q = question.strip().lower()

    if "දුම්රිය" in q or "කෝච්චි" in q:
        return "train_services"

    if "නවාතැන්" in q or "හෝටල්" in q:
        return "accommodation"

    if "කඳු නැග" in q or "තරණය" in q:
        return "hiking"

    if "පළාත" in q:
        return "province"

    if "දිස්ත්‍රික්ක" in q:
        return "district"

    if any(word in q for word in [
        "සුදුසු කාල", "සුදුසු වේලාව",
        "හොඳම වෙලාව", "හොඳම කාල",
        "යන්න හොඳ", "නැරඹීමට සුදුසු"
    ]):
        return "visiting_time"

    if "උස" in q:
        if any(word in q for word in [
            "සහ", "අතරින්", "වැඩි", "සැසඳ", "වඩා"
        ]):
            return "height_comparison"
        return "height"

    return "unknown"


def identify_entities(question):
    q = question.lower()

    matches = []
    occupied = []

    candidates = []

    for entity_id, aliases in ENTITY_ALIASES.items():
        for alias in aliases:
            candidates.append((entity_id, alias))

    # Match longer names first to avoid overlapping names
    candidates.sort(key=lambda x: len(x[1]), reverse=True)

    for entity_id, alias in candidates:
        start = q.find(alias.lower())

        if start == -1:
            continue

        end = start + len(alias)

        # Avoid overlapping entity matches
        if any(start < e and end > s for s, e in occupied):
            continue

        if entity_id not in [m["id"] for m in matches]:
            matches.append({
                "id": entity_id,
                "matched_text": alias
            })
            occupied.append((start, end))

    # Return entities in their original question order
    matches.sort(
        key=lambda m: q.find(m["matched_text"].lower())
    )

    return matches


def analyze_question(question):
    return {
        "intent": identify_intent(question),
        "entities": identify_entities(question)
    }


print("Sinhala question-processing functions loaded successfully!")

Sinhala question-processing functions loaded successfully!


Test adaptive evidence retrieval

In [24]:
for question_id, item in GOLD_EVIDENCE.items():

    result = adaptive_evidence_retrieve(
        item["question"]
    )

    print("\n" + "=" * 55)
    print("Question:", question_id)
    print("Status:", result["status"])
    print("Intent:", result["intent"])
    print("Entities:", result["entity_ids"])
    print("Retrieved facts:", len(result["facts"]))

    print(
        "Gold evidence complete:",
        item["facts"].issubset(result["facts"])
    )


Question: Q001
Status: success
Intent: province
Entities: ['D001']
Retrieved facts: 2
Gold evidence complete: True

Question: Q002
Status: success
Intent: accommodation
Entities: ['D001']
Retrieved facts: 4
Gold evidence complete: True

Question: Q003
Status: success
Intent: hiking
Entities: ['D001']
Retrieved facts: 4
Gold evidence complete: True

Question: Q004
Status: success
Intent: height
Entities: ['A004']
Retrieved facts: 1
Gold evidence complete: True

Question: Q005
Status: success
Intent: height_comparison
Entities: ['A004', 'A007']
Retrieved facts: 2
Gold evidence complete: True

Question: Q006
Status: success
Intent: visiting_time
Entities: ['A001']
Retrieved facts: 1
Gold evidence complete: True


Compare all five methods
Now we'll combine:
1. 0-hop
2. 1-hop
3. 2-hop
4. Relationship-aware
5. Rule-based adaptive

In [25]:
comparison_rows = []

for question_id, item in GOLD_EVIDENCE.items():

    methods = {}

    # Fixed-hop methods
    for depth in [0, 1, 2]:
        evidence = get_graph_evidence(
            item["entities"],
            max_hops=depth
        )

        methods[f"{depth}-hop"] = normalize_evidence(
            evidence
        )

    # Relationship-aware: oracle intent/entities
    methods["Relationship-aware"] = (
        targeted_evidence_retrieve(
            item["intent"],
            item["entities"]
        )
    )

    # Adaptive: automatic question analysis
    adaptive_result = adaptive_evidence_retrieve(
        item["question"]
    )

    methods["Adaptive"] = adaptive_result["facts"]

    for method_name, facts in methods.items():

        scores = score_evidence(
            retrieved_facts=facts,
            required_facts=item["facts"]
        )

        comparison_rows.append({
            "Question_ID": question_id,
            "Method": method_name,
            **scores
        })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,Question_ID,Method,precision,recall,f1,complete,retrieved_count,matched_count
0,Q001,0-hop,0.000000,0.0,0.000000,False,2,0
1,Q001,1-hop,0.018519,0.5,0.035714,False,54,1
2,Q001,2-hop,0.026667,1.0,0.051948,True,75,2
3,Q001,Relationship-aware,1.000000,1.0,1.000000,True,2,2
4,Q001,Adaptive,1.000000,1.0,1.000000,True,2,2
5,Q002,0-hop,0.000000,0.0,0.000000,False,2,0
6,Q002,1-hop,0.074074,1.0,0.137931,True,54,4
7,Q002,2-hop,0.053333,1.0,0.101266,True,75,4
8,Q002,Relationship-aware,1.000000,1.0,1.000000,True,4,4
9,Q002,Adaptive,1.000000,1.0,1.000000,True,4,4


Generate the five-method summary

In [26]:
comparison_summary = (
    comparison_df
    .groupby("Method", as_index=False)
    .agg(
        Mean_Precision=("precision", "mean"),
        Mean_Recall=("recall", "mean"),
        Mean_F1=("f1", "mean"),
        Complete_Questions=("complete", "sum"),
        Mean_Facts=("retrieved_count", "mean")
    )
)

comparison_summary = comparison_summary.sort_values(
    "Mean_F1",
    ascending=False
)

display(comparison_summary.round(3))

,Method,Mean_Precision,Mean_Recall,Mean_F1,Complete_Questions,Mean_Facts
4,Relationship-aware,1.000,1.000,1.000,6,2.333
3,Adaptive,1.000,1.000,1.000,6,2.333
0,0-hop,0.103,0.500,0.170,3,4.167
1,1-hop,0.087,0.833,0.155,4,32.000
2,2-hop,0.034,1.000,0.066,6,64.833


In [27]:
output_dir = project_root / "results"
output_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    output_dir / "five_method_comparison_details.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_summary.to_csv(
    output_dir / "five_method_comparison_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Five-method comparison saved successfully!")

Five-method comparison saved successfully!


Repeated latency testing

Main finding
Both relationship-aware and adaptive retrieval achieved perfect evidence scores on the six development questions.

However, this does not yet prove that they are the best methods for unseen questions, because the targeted retrieval rules and expected evidence were developed together.

Also, both approaches currently use the same targeted evidence queries. Their difference is how the query is selected:
- Relationship-aware: Correct intent and entities are supplied manually.
- Adaptive: Intent and entities are identified automatically from the Sinhala question.
Their identical scores are therefore expected when automatic identification succeeds.

Measure execution time for all five methods

In [28]:
import time
import statistics
import pandas as pd

def run_retrieval_method(method, item):

    if method in ["0-hop", "1-hop", "2-hop"]:
        depth = int(method[0])

        evidence = get_graph_evidence(
            item["entities"],
            max_hops=depth
        )

        return normalize_evidence(evidence)

    elif method == "Relationship-aware":
        return targeted_evidence_retrieve(
            item["intent"],
            item["entities"]
        )

    elif method == "Adaptive":
        return adaptive_evidence_retrieve(
            item["question"]
        )["facts"]

    raise ValueError(f"Unknown method: {method}")


methods = [
    "0-hop",
    "1-hop",
    "2-hop",
    "Relationship-aware",
    "Adaptive"
]

latency_rows = []

for question_id, item in GOLD_EVIDENCE.items():

    for method in methods:

        # Warm-up
        for _ in range(5):
            run_retrieval_method(method, item)

        times_ms = []

        for _ in range(30):
            start = time.perf_counter()

            run_retrieval_method(method, item)

            elapsed_ms = (
                time.perf_counter() - start
            ) * 1000

            times_ms.append(elapsed_ms)

        latency_rows.append({
            "Question_ID": question_id,
            "Method": method,
            "Median_ms": statistics.median(times_ms),
            "Mean_ms": statistics.mean(times_ms),
            "P95_ms": pd.Series(times_ms).quantile(0.95)
        })

latency_df = pd.DataFrame(latency_rows)

display(latency_df)

,Question_ID,Method,Median_ms,Mean_ms,P95_ms
0,Q001,0-hop,32.32635,32.625343,42.617745
1,Q001,1-hop,35.63460,38.182447,53.020285
2,Q001,2-hop,34.57750,36.556243,46.280685
3,Q001,Relationship-aware,15.38105,16.527077,18.464270
4,Q001,Adaptive,14.74490,15.367177,18.981105
5,Q002,0-hop,25.36230,26.349920,32.740750
6,Q002,1-hop,33.59650,34.402077,43.169485
7,Q002,2-hop,40.74710,41.701953,48.880900
8,Q002,Relationship-aware,15.18725,16.393297,22.616645
9,Q002,Adaptive,11.73500,11.958197,13.740815


Summarize latency across all questions

In [29]:
latency_summary = (
    latency_df
    .groupby("Method", as_index=False)
    .agg(
        Median_Latency_ms=("Median_ms", "median"),
        Mean_Latency_ms=("Mean_ms", "mean"),
        P95_Latency_ms=("P95_ms", "mean")
    )
    .sort_values("Median_Latency_ms")
)

display(latency_summary.round(3))

,Method,Median_Latency_ms,Mean_Latency_ms,P95_Latency_ms
4,Relationship-aware,10.111,12.690,20.099
3,Adaptive,10.496,11.201,14.620
0,0-hop,23.192,24.045,29.993
1,1-hop,25.318,26.757,37.098
2,2-hop,27.082,31.699,38.516


In [30]:
latency_df.to_csv(
    output_dir / "retrieval_latency_details.csv",
    index=False,
    encoding="utf-8-sig"
)

latency_summary.to_csv(
    output_dir / "retrieval_latency_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Latency evaluation saved successfully!")

Latency evaluation saved successfully!


Final unseen Sinhala question evaluation

Create the unseen question dataset

In [31]:
UNSEEN_QUESTIONS = [
    {
        "id": "U01",
        "question": "ඇල්ල අයත් වන පළාතේ නම කියන්න.",
        "intent": "province",
        "entities": ["D001"],
        "gold_id": "Q001"
    },
    {
        "id": "U02",
        "question": "ඇල්ලේ රැඳී සිටීමට හෝටල් තිබෙන්නේ මොනවාද?",
        "intent": "accommodation",
        "entities": ["D001"],
        "gold_id": "Q002"
    },
    {
        "id": "U03",
        "question": "ඇල්ලේ තරණය කිරීමට හැකි කඳු මොනවාද?",
        "intent": "hiking",
        "entities": ["D001"],
        "gold_id": "Q003"
    },
    {
        "id": "U04",
        "question": "රාවණා ඇල්ල මීටර් කීයක් උස වෙනවාද?",
        "intent": "height",
        "entities": ["A004"],
        "gold_id": "Q004"
    },
    {
        "id": "U05",
        "question": "දියලුම ඇල්ල රාවණා ඇල්ලට වඩා උසද?",
        "intent": "height_comparison",
        "entities": ["A007", "A004"],
        "gold_id": "Q005"
    },
    {
        "id": "U06",
        "question": "පුංචි ශ්‍රී පාදයේ සංචාරයට හොඳ වෙලාව කුමක්ද?",
        "intent": "visiting_time",
        "entities": ["A001"],
        "gold_id": "Q006"
    },
    {
        "id": "U07",
        "question": "ඇල්ලට නවාතැන් ගන්න තියෙන ස්ථාන සඳහන් කරන්න.",
        "intent": "accommodation",
        "entities": ["D001"],
        "gold_id": "Q002"
    },
    {
        "id": "U08",
        "question": "ඇල්ලේ කඳු තරණය කරන්න පුළුවන් තැන් මොනවාද?",
        "intent": "hiking",
        "entities": ["D001"],
        "gold_id": "Q003"
    }
]

print("Unseen supported questions:", len(UNSEEN_QUESTIONS))

Unseen supported questions: 8


Evaluate all five methods on unseen questions

In [32]:
unseen_rows = []

for item in UNSEEN_QUESTIONS:

    gold_facts = GOLD_EVIDENCE[item["gold_id"]]["facts"]

    methods = {}

    for depth in [0, 1, 2]:
        evidence = get_graph_evidence(
            item["entities"],
            max_hops=depth
        )
        methods[f"{depth}-hop"] = normalize_evidence(evidence)

    methods["Relationship-aware"] = targeted_evidence_retrieve(
        item["intent"],
        item["entities"]
    )

    adaptive_result = adaptive_evidence_retrieve(
        item["question"]
    )

    methods["Adaptive"] = adaptive_result["facts"]

    for method, facts in methods.items():
        scores = score_evidence(facts, gold_facts)

        unseen_rows.append({
            "Question_ID": item["id"],
            "Method": method,
            "Expected_Intent": item["intent"],
            "Predicted_Intent": (
                adaptive_result["intent"]
                if method == "Adaptive" else item["intent"]
            ),
            "Status": (
                adaptive_result["status"]
                if method == "Adaptive" else "oracle"
            ),
            **scores
        })

unseen_df = pd.DataFrame(unseen_rows)

unseen_summary = (
    unseen_df.groupby("Method", as_index=False)
    .agg(
        Mean_Precision=("precision", "mean"),
        Mean_Recall=("recall", "mean"),
        Mean_F1=("f1", "mean"),
        Complete_Questions=("complete", "sum"),
        Mean_Facts=("retrieved_count", "mean")
    )
    .sort_values("Mean_F1", ascending=False)
)

display(unseen_summary.round(3))

,Method,Mean_Precision,Mean_Recall,Mean_F1,Complete_Questions,Mean_Facts
4,Relationship-aware,1.000,1.000,1.000,8,2.750
3,Adaptive,0.875,0.875,0.875,7,2.625
1,1-hop,0.079,0.812,0.142,5,37.500
0,0-hop,0.077,0.375,0.127,3,3.625
2,2-hop,0.039,1.000,0.075,8,67.375


In [33]:
adaptive_failures = unseen_df[
    (unseen_df["Method"] == "Adaptive")
    & (~unseen_df["complete"])
]

display(adaptive_failures)

print(
    "Adaptive complete answers:",
    8 - len(adaptive_failures),
    "/ 8"
)

,Question_ID,Method,Expected_Intent,Predicted_Intent,Status,precision,recall,f1,complete,retrieved_count,matched_count
29,U06,Adaptive,visiting_time,unknown,unsupported,0.0,0.0,0.0,False,0,0


Adaptive complete answers: 7 / 8


Test unsupported questions

In [34]:
UNSUPPORTED_QUESTIONS = [
    "ඇල්ලේ ජනගහනය කොපමණද?",
    "සීගිරියේ උස මීටර් කීයද?",
    "ඇල්ලේ පාසල් ගණන කීයද?"
]

for question in UNSUPPORTED_QUESTIONS:

    result = adaptive_evidence_retrieve(question)

    print("\nQuestion:", question)
    print("Status:", result["status"])
    print("Retrieved facts:", len(result["facts"]))


Question: ඇල්ලේ ජනගහනය කොපමණද?
Status: unsupported
Retrieved facts: 0

Question: සීගිරියේ උස මීටර් කීයද?
Status: invalid_entities
Retrieved facts: 0

Question: ඇල්ලේ පාසල් ගණන කීයද?
Status: unsupported
Retrieved facts: 0


In [35]:
unseen_df.to_csv(
    output_dir / "unseen_question_details.csv",
    index=False,
    encoding="utf-8-sig"
)

unseen_summary.to_csv(
    output_dir / "unseen_question_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Unseen question evaluation saved!")

Unseen question evaluation saved!


Display Sinhala answers from Neo4j

Create a Sinhala answer formatter

In [36]:
def format_sinhala_answer(intent, records):

    if not records:
        return "මෙම ප්‍රශ්නයට පිළිතුරු ලබා දීමට ප්‍රමාණවත් තොරතුරු නොමැත."

    if intent == "province":
        return f"ඇල්ල පිහිටා ඇත්තේ {records[0]['province']} තුළයි."

    elif intent == "district":
        return f"ඇල්ල පිහිටා ඇත්තේ {records[0]['district']} දිස්ත්‍රික්කයේය."

    elif intent == "accommodation":
        hotels = [r["accommodation"] for r in records]
        return "ඇල්ලේ නවාතැන් පහසුකම් අතර " + ", ".join(hotels) + " ඇතුළත් වේ."

    elif intent == "hiking":
        locations = [r["hiking_location"] for r in records]
        return "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන අතර " + ", ".join(locations) + " ඇතුළත් වේ."

    elif intent == "height":
        r = records[0]
        qualifier = "ආසන්න වශයෙන් " if r.get("approximate") else ""
        return f"{r['name']} උස {qualifier}මීටර් {r['height_m']} කි."

    elif intent == "height_comparison":
        r = records[0]

        if r["first_height"] > r["second_height"]:
            taller = r["first_name"]
        elif r["first_height"] < r["second_height"]:
            taller = r["second_name"]
        else:
            return "මෙම ස්ථාන දෙකේ උස සමාන වේ."

        return (
            f"{r['first_name']} මීටර් {r['first_height']}ක් සහ "
            f"{r['second_name']} මීටර් {r['second_height']}ක් උසයි. "
            f"එබැවින් {taller} උසින් වැඩිය."
        )

    elif intent == "train_services":
        trains = [
            f"{r['train']} (අංක {r['train_number']})"
            for r in records
        ]
        return "ඇල්ලට පැමිණෙන දුම්රිය සේවා: " + ", ".join(trains) + "."

    elif intent == "visiting_time":
        r = records[0]
        return f"{r['name']} නැරඹීමට සුදුසු කාලය {r['visiting_time']} වේ."

    return "ලබාගත් තොරතුරු පිළිතුරක් ලෙස සකස් කළ නොහැකි විය."

Connect retrieval to answer formatting

In [37]:
def ask_kg(question):

    analysis = analyze_question(question)

    intent = analysis["intent"]

    entity_ids = [
        e["id"]
        for e in analysis["entities"]
    ]

    if intent == "unknown":
        return "මෙම ප්‍රශ්නය තවම හඳුනාගත නොහැකි විය."

    result = adaptive_retrieve(intent, entity_ids)

    if result["status"] != "success":
        return "Knowledge Graph තුළ මෙම ප්‍රශ්නයට අදාළ තොරතුරු සොයාගත නොහැකි විය."

    return format_sinhala_answer(
        intent,
        result["records"]
    )

Sinhala question
        |
Intent + entity identification
        |
Adaptive Neo4j retrieval
        |
Sinhala answer formatting

Test questions and display answers

In [38]:
test_questions = [
    "ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?",
    "ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?",
    "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?",
    "රාවණා ඇල්ලේ උස කීයද?",
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
    "ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?",
    "පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?"
]

for i, question in enumerate(test_questions, 1):

    print(f"\n{'=' * 65}")
    print(f"QUESTION {i}: {question}")

    answer = ask_kg(question)

    print(f"ANSWER: {answer}")


QUESTION 1: ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?
ANSWER: ඇල්ල පිහිටා ඇත්තේ ඌව පළාත තුළයි.

QUESTION 2: ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?
ANSWER: ඇල්ලේ නවාතැන් පහසුකම් අතර 98 Acres Resort & Spa, Ella Jungle Resort, La Ella Breeze, The Secret Ella ඇතුළත් වේ.

QUESTION 3: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?
ANSWER: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන අතර ඇල්ල රොක්, පුංචි ශ්‍රී පාදය ඇතුළත් වේ.

QUESTION 4: රාවණා ඇල්ලේ උස කීයද?
ANSWER: රාවණා ඇල්ල උස ආසන්න වශයෙන් මීටර් 25 කි.

QUESTION 5: රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?
ANSWER: රාවණා ඇල්ල මීටර් 25ක් සහ දියලුම දිය ඇල්ල මීටර් 220ක් උසයි. එබැවින් දියලුම දිය ඇල්ල උසින් වැඩිය.

QUESTION 6: ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?
ANSWER: ඇල්ලට පැමිණෙන දුම්රිය සේවා: පොඩි මැණිකේ (අංක 1005), දුම්රිය අංක 1007 (අංක 1007), උඩරට මැණිකේ (අංක 1015), රාත්‍රී තැපැල් දුම්රිය (අංක 1045).

QUESTION 7: පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?
ANSWER: පුංචි ශ්‍රී පාදය නැරඹීමට සුදුසු කාලය උදෑසන සහ සවස් කාලය වේ.


In [39]:
question = "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?"

print("Question:", question)
print("Answer:", ask_kg(question))

Question: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?
Answer: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන අතර ඇල්ල රොක්, පුංචි ශ්‍රී පාදය ඇතුළත් වේ.
